In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics flwr[simulation]

In [ ]:
!pip install ultralytics flwr[simulation] roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="LpOBfSxnkvicemyEuHPG")
project = rf.workspace("weapon-detection-cctv").project("weapon-detection-cctv-v3-dataset")
version = project.version(1)
dataset = version.download("yolov8")

In [ ]:
# This script splits a dataset of images and labels into two equal parts for two clients,
# creating separate directory structures and copying corresponding files along with the config file.

import os
import shutil

dataset_path = "/content/weapon-detection-cctv-v3-dataset-1"
clients = ["client_1", "client_2"]

for client in clients:
    os.makedirs(f"{client}/train/images", exist_ok=True)
    os.makedirs(f"{client}/train/labels", exist_ok=True)
    shutil.copy(f"{dataset_path}/data.yaml", f"{client}/data.yaml")

images = os.listdir(f"{dataset_path}/train/images")
midpoint = len(images) // 2

for i, img in enumerate(images):
    label = img.replace(".jpg", ".txt").replace(".png", ".txt")
    target_client = "client_1" if i < midpoint else "client_2"

    shutil.copy(f"{dataset_path}/train/images/{img}", f"{target_client}/train/images/{img}")
    if os.path.exists(f"{dataset_path}/train/labels/{label}"):
        shutil.copy(f"{dataset_path}/train/labels/{label}", f"{target_client}/train/labels/{label}")

print("Data successfully split! Client 1 and Client 2 are ready.")

In [ ]:
# This script locates the dataset folder, reads its YAML configuration,
# and updates the training paths for two client datasets while keeping a shared validation set.

import yaml
import glob
import os

dataset_dirs = glob.glob('/content/weapon-detection-cctv-*')

if dataset_dirs:
    original_dataset_path = dataset_dirs[0]

    with open(f'{original_dataset_path}/data.yaml', 'r') as f:
        data = yaml.safe_load(f)

    data['train'] = '/content/client_1/train/images'
    data['val'] = f'{original_dataset_path}/valid/images'

    if 'test' in data:
        del data['test']

    with open('/content/client_1/data.yaml', 'w') as f:
        yaml.dump(data, f)

    data['train'] = '/content/client_2/train/images'

    with open('/content/client_2/data.yaml', 'w') as f:
        yaml.dump(data, f)

In [ ]:
# This script initializes a YOLOv8 model and trains it for one epoch on Client 1's dataset to verify the training pipeline.

from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(data='client_1/data.yaml', epochs=1, imgsz=640)

In [ ]:
# This script performs federated learning using YOLOv8 by training local models on multiple clients,
# averaging their weights (FedAvg), and updating a global model iteratively across several rounds.

import torch
from ultralytics import YOLO
import copy
import os

init_model = YOLO('yolov8n.pt')

init_model.train(
    data='/content/client_1/data.yaml',
    epochs=5,
    imgsz=640,
    project='/content/FL_Server',
    name='global_init',
    exist_ok=True,
    workers=0
)

global_model_path = '/content/FL_Server/global_init/weights/best.pt'

NUM_ROUNDS = 10
CLIENTS = ['/content/client_1/data.yaml', '/content/client_2/data.yaml']

for round_num in range(NUM_ROUNDS):

    client_model_paths = []

    for i, client_data in enumerate(CLIENTS):

        local_model = YOLO(global_model_path)

        client_proj = f'/content/FL_Clients/Round_{round_num}'
        client_name = f'ATM_{i+1}'

        local_model.train(
            data=client_data,
            epochs=5,
            imgsz=640,
            project=client_proj,
            name=client_name,
            exist_ok=True,
            workers=0
        )

        client_model_paths.append(f'{client_proj}/{client_name}/weights/best.pt')

    state_dicts = [torch.load(path, weights_only=False)['model'].state_dict() for path in client_model_paths]

    avg_state_dict = {}
    for key in state_dicts[0].keys():
        stacked = torch.stack([sd[key].float() for sd in state_dicts])
        avg_state_dict[key] = torch.mean(stacked, dim=0).to(state_dicts[0][key].dtype)

    global_checkpoint = torch.load(global_model_path, weights_only=False)
    global_checkpoint['model'].load_state_dict(avg_state_dict)
    torch.save(global_checkpoint, global_model_path)

print(global_model_path)

In [ ]:
# This script loads the trained federated YOLO model, performs inference on a validation image,
# and displays the predicted output with bounding boxes.

from ultralytics import YOLO
import glob
from IPython.display import Image, display
import os

model = YOLO('/content/FL_Server/global_init/weights/best.pt')

val_images = glob.glob('/content/weapon-detection-*/valid/images/*.jpg')

if val_images:
    test_image = val_images[0]

    results = model.predict(source=test_image, save=True, conf=0.10, imgsz=1280, augment=True)

    latest_predict_folder = max(glob.glob('/content/runs/detect/predict*'), key=os.path.getmtime)
    image_name = os.path.basename(test_image)

    display(Image(filename=f'{latest_predict_folder}/{image_name}'))

In [ ]:
# Run the trained federated model on a few random validation images and show predictions

from ultralytics import YOLO
import glob
import random
import os
from IPython.display import Image, display

model = YOLO('/content/FL_Server/global_init/weights/best.pt')

val_images = glob.glob('/content/weapon-detection-*/valid/images/*.jpg')

if len(val_images) >= 3:
    samples = random.sample(val_images, 3)

    for img_path in samples:
        print(f"\nTesting: {os.path.basename(img_path)}")

        model.predict(source=img_path, save=True, conf=0.10, augment=True)

        latest_folder = max(glob.glob('/content/runs/detect/predict*'), key=os.path.getmtime)
        display(Image(filename=f"{latest_folder}/{os.path.basename(img_path)}"))

In [ ]:
# Change your project path in the training cell to:
project='/content/drive/MyDrive/EE655_Project'

In [ ]:
# Run inference on a video file
results = model.predict(
    source='/content/💰 Man Tries to Break Open ATM _ Caught on CCTV _ Real Crime USA 🚨_shorts _cctv _cctvcamera _crime(720P_HD).mp4',
    save=True,
    conf=0.20,
    imgsz=1280,
    vid_stride=1,
    stream=False
)

In [ ]:
# Capture an image from the webcam and run the trained model on it

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
from ultralytics import YOLO

model = YOLO('/content/FL_Server/global_init/weights/best.pt')

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);

            video.srcObject = stream;
            await video.play();

            return new Promise((resolve) => {
                setTimeout(() => {
                    const canvas = document.createElement('canvas');
                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    canvas.getContext('2d').drawImage(video, 0, 0);

                    stream.getVideoTracks()[0].stop();
                    div.remove();

                    resolve(canvas.toDataURL('image/jpeg', quality));
                }, 5000);
            });
        }
    ''')

    display(js)
    data = eval_js(f'takePhoto({quality})')
    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

try:
    img_path = take_photo()

    results = model.predict(source=img_path, conf=0.15, imgsz=1280, save=True)

    from PIL import Image
    res_img = results[0].plot()
    display(Image.fromarray(res_img[:, :, ::-1]))

except Exception as err:
    print(err)

In [ ]:
import time

for r in results:
    if r.boxes is not None:
        for box in r.boxes:
            if int(box.cls[0]) == 1:
                conf = float(box.conf[0])

                if conf > 0.4:
                    print("HIGH CONFIDENCE WEAPON!")

                elif conf > 0.2:
                    print("Possible weapon detected")

In [ ]:
from ultralytics import YOLO
import os

model = YOLO('/content/best-3.pt')
print("Global Brain Loaded")

In [ ]:
# Inference with High-Res to catch small tools/weapons
results = model.predict(
    source='/content/Surveillance_ Thieves rip open ATM in Shelbyville(720P_HD).mp4',
    save=True,
    conf=0.15,      # Lowered slightly to catch smaller objects
    imgsz=1280,     # Essential for high-detail CCTV
    stream=False    # Ensures the file is written to disk properly
)

print("Inference complete!")

In [ ]:
for r in results:
    classes = r.boxes.cls.tolist()

    # Check if both Class 0 (Person) and Class 1 (Weapon) are in the same frame
    if 0 in classes and 1 in classes:
        # Check confidence of the weapon
        weapon_indices = [i for i, x in enumerate(classes) if x == 1]
        highest_weapon_conf = max([r.boxes.conf[i] for i in weapon_indices])

        if highest_weapon_conf > 0.25:
            print("CRITICAL ALERT: Armed Person Detected in ATM!")
    else:
        print("Situation Normal")

In [ ]:
# Extract frames from burglary videos, auto-label them using a pretrained model,
# and fine-tune a YOLO model on the generated dataset

import os
import glob
import cv2
import shutil
from ultralytics import YOLO

VIDEO_DIR = '/content/Burglary_Data'
IMAGES_DIR = '/content/datasets/burglary_frames/train/images'
LABELS_DIR = '/content/datasets/burglary_frames/train/labels'

if os.path.exists('/content/datasets'):
    shutil.rmtree('/content/datasets')

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)

video_files = glob.glob(os.path.join(VIDEO_DIR, '**/*.mp4'), recursive=True)

if video_files:
    for v_path in video_files[:20]:
        cap = cv2.VideoCapture(v_path)
        v_name = os.path.basename(v_path).split('.')[0]
        count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if count % 45 == 0:
                cv2.imwrite(f"{IMAGES_DIR}/{v_name}_f{count}.jpg", frame)

            count += 1

        cap.release()

if os.listdir(IMAGES_DIR):
    model = YOLO('/content/best-3.pt')

    model.predict(source=IMAGES_DIR, conf=0.05, save_txt=True, imgsz=1280)

    predict_labels = glob.glob('/content/runs/detect/predict*/labels/*.txt')

    if predict_labels:
        for label_path in predict_labels:
            shutil.move(label_path, os.path.join(LABELS_DIR, os.path.basename(label_path)))

if os.listdir(LABELS_DIR):
    model.train(
        data='/content/burglary_data.yaml',
        epochs=30,
        imgsz=1280,
        batch=4,
        cls=2.5,
        project='ATM_Final_Model',
        name='v1'
    )

In [ ]:
# Run the trained model on a video and print confidence stats for detected objects

from ultralytics import YOLO
import cv2

model = YOLO('/content/runs/detect/ATM_Final_Model/v1/weights/best.pt')

video_source = '/content/Burglary_Data/Burglary001_x264A.mp4'
results = model.predict(source=video_source, save=True, conf=0.25, imgsz=1280)

print("\n--- CONFIDENCE ANALYSIS ---")

for i, r in enumerate(results[:10]):
    if len(r.boxes) > 0:
        conf_scores = r.boxes.conf.tolist()
        avg_conf = sum(conf_scores) / len(conf_scores)
        max_conf = max(conf_scores)

        print(f"Frame {i}: {len(r.boxes)} objects | Max: {max_conf:.2f} | Avg: {avg_conf:.2f}")

        if max_conf > 0.70:
            print(f"High confidence detection ({max_conf:.2f})")

In [ ]:
# Process a video frame-by-frame, run detections, and display annotated output in a loop

import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from IPython.display import clear_output

model_path = '/content/runs/detect/ATM_Final_Model/v1/weights/best.pt'
model = YOLO(model_path)

video_path = 'input_label.mp4'
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Processing: {video_path} ({width}x{height} @ {fps}fps)")

frame_count = 0

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1

        if frame_count % 3 == 0:
            results = model.predict(source=frame, conf=0.25, imgsz=640, verbose=False)

            annotated_frame = results[0].plot()

            clear_output(wait=True)

            display_frame = cv2.resize(annotated_frame, (800, 600))
            cv2_imshow(display_frame)

            print(f"Frame: {frame_count} | Detections: {len(results[0].boxes)}")

except KeyboardInterrupt:
    print("Stopped")

cap.release()
print("Done")

In [ ]:
# Build a quick evaluation set from video frames, label it, and compare two models

import os
import zipfile
import glob
import cv2
import shutil
import pandas as pd
from ultralytics import YOLO

if os.path.exists('/content/Burglary.zip'):
    with zipfile.ZipFile('/content/Burglary.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/Burglary_Data')

IMAGES_DIR = '/content/eval_data/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

video_files = glob.glob('/content/Burglary_Data/**/*.mp4', recursive=True)

for v_path in video_files[:10]:
    cap = cv2.VideoCapture(v_path)
    v_name = os.path.basename(v_path).split('.')[0]
    count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if count % 60 == 0:
            cv2.imwrite(f"{IMAGES_DIR}/{v_name}_f{count}.jpg", frame)

        count += 1

    cap.release()

model_gt = YOLO('/content/best.pt')
model_gt.predict(source=IMAGES_DIR, conf=0.15, save_txt=True, imgsz=1280)

LABELS_DIR = '/content/eval_data/labels'
os.makedirs(LABELS_DIR, exist_ok=True)

latest_predict = max(glob.glob('/content/runs/detect/predict*/'), key=os.path.getmtime)

for label_file in glob.glob(f"{latest_predict}/labels/*.txt"):
    shutil.move(label_file, f"{LABELS_DIR}/{os.path.basename(label_file)}")

with open('/content/eval_data.yaml', 'w') as f:
    f.write(
        "path: /content/eval_data\n"
        "train: images\n"
        "val: images\n"
        "names:\n  0: person\n  1: weapon"
    )

models_to_compare = {
    "Base Model": "/content/best-3.pt",
    "Tuned Model": "/content/best.pt"
}

results_list = []

for name, path in models_to_compare.items():
    print(f"Evaluating {name}...")

    model = YOLO(path)
    metrics = model.val(data='/content/eval_data.yaml', imgsz=1280, verbose=False)

    results_list.append({
        "Model": name,
        "mAP@50": metrics.box.map50,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "Fitness": metrics.fitness
    })

comparison_df = pd.DataFrame(results_list)

print("\n" + "="*50)
print(comparison_df.to_string(index=False))

In [ ]:
!pip install gradio -q

import gradio as gr
import cv2
from ultralytics import YOLO
import tempfile
import os

model = YOLO('/content/best.pt')

def predict_video(video_path):
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    output_path = tempfile.mktemp(suffix='.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print("Processing started!!")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLO inference
        results = model.predict(frame, conf=0.25, imgsz=640, verbose=False)
        annotated_frame = results[0].plot()
        out.write(annotated_frame)

    cap.release()
    out.release()
    print("Processing complete!!")
    return output_path
demo = gr.Interface(
    fn=predict_video,
    inputs=gr.Video(label="Upload ATM Footage (.mp4)"),
    outputs=gr.Video(label="AI Detection Result"),
    title="",
    description="Upload surveillance footage to automatically identify individuals and detect possible weapons using a trained deep learning model. The system analyzes each frame and highlights relevant objects to assist with rapid security assessment.",
    theme="soft"
)
demo.launch(share=True, debug=True)